# Imports & paths

In [1]:
import pandas as pd
from pathlib import Path

DATA_INTERIM = Path("../data/interim")
DATA_PROCESSED = Path("../data/processed")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Load clean data

In [2]:
df_customers = pd.read_parquet(DATA_INTERIM / "customers_clean.parquet")
df_txn = pd.read_parquet(DATA_INTERIM / "transactions_clean.parquet")

# Create RFM

In [3]:
snapshot_date = df_txn["transaction_date"].max() + pd.Timedelta(days=1)
snapshot_date

Timestamp('2026-01-01 00:00:00')

In [4]:
rfm = (
    df_txn
    .groupby("customer_id")
    .agg({
        "transaction_date": lambda x: (snapshot_date - x.max()).days,
        "customer_id": "count",
        "amount": "sum"
    })
    .rename(columns={
        "transaction_date": "recency",
        "customer_id": "frequency",
        "amount": "monetary"
    })
    .reset_index()
)

rfm.head()


,customer_id,recency,frequency,monetary
0,C00000,1,12,1222.85
1,C00001,12,19,1228.46
2,C00002,97,11,910.64
3,C00003,44,4,114.71
4,C00004,109,19,2018.94


## 1. Quantile-Based RFM Score

In [5]:
rfm_q = rfm.copy()


rfm_q["R_score"] = pd.qcut(
    rfm_q["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
)


rfm_q["F_score"] = pd.qcut(
    rfm_q["frequency"],
    q=5,
    labels=[1, 2, 3, 4, 5]
)


rfm_q["M_score"] = pd.qcut(
    rfm_q["monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

rfm_q[["customer_id", "R_score", "F_score", "M_score"]].head()

,customer_id,R_score,F_score,M_score
0,C00000,5,3,4
1,C00001,4,4,4
2,C00002,2,3,4
3,C00003,3,1,1
4,C00004,2,4,5


In [6]:
rfm_q["RFM_score"] = (
    rfm_q["R_score"].astype(str) +
    rfm_q["F_score"].astype(str) +
    rfm_q["M_score"].astype(str)
)

rfm_q[["customer_id", "RFM_score"]].head()

,customer_id,RFM_score
0,C00000,534
1,C00001,444
2,C00002,234
3,C00003,311
4,C00004,245


In [7]:
def rfm_segment(row):
    r, f, m = int(row["R_score"]), int(row["F_score"]), int(row["M_score"])

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    if r >= 4 and f >= 3:
        return "Loyal Customers"

    if r >= 4 and f <= 2:
        return "New Customers"

    if r <= 2 and f >= 3:
        return "At Risk"

    if r <= 2 and f <= 2:
        return "Hibernating"

    return "Others"

rfm_q["rfm_segment"] = rfm_q.apply(rfm_segment, axis=1)
rfm_q[["customer_id", "RFM_score", "rfm_segment"]].head()

,customer_id,RFM_score,rfm_segment
0,C00000,534,Loyal Customers
1,C00001,444,Champions
2,C00002,234,At Risk
3,C00003,311,Others
4,C00004,245,At Risk


In [8]:
rfm_q["rfm_segment"].value_counts()

rfm_segment
Hibernating        681
Champions          578
Others             576
At Risk            473
Loyal Customers    368
New Customers      216
Name: count, dtype: int64

## 2. KMeans

In [9]:
from sklearn.preprocessing import StandardScaler

features = ["recency", "frequency", "monetary"]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[features])

rfm_scaled_df = pd.DataFrame(
    rfm_scaled,
    columns=[f"{c}_scaled" for c in features]
)

rfm_scaled_df.head()

,recency_scaled,frequency_scaled,monetary_scaled
0,-0.958971,-0.242153,0.233769
1,-0.838010,0.166359,0.238421
2,0.096691,-0.300511,-0.025108
3,-0.486122,-0.709023,-0.685074
4,0.228648,0.166359,0.893867


In [10]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

rfm["segment"] = kmeans.fit_predict(rfm_scaled)

rfm.head()

,customer_id,recency,frequency,monetary,segment
0,C00000,1,12,1222.85,3
1,C00001,12,19,1228.46,3
2,C00002,97,11,910.64,3
3,C00003,44,4,114.71,3
4,C00004,109,19,2018.94,0


### 2.1 Evaluation 

In [11]:
segment_profile = (
    rfm
    .groupby("segment")[features]
    .mean()
)

segment_profile

,recency,frequency,monetary
segment,,,
0,35.150000,32.182143,1953.514554
1,208.908033,7.677532,431.335448
2,12.063063,73.450450,5052.564775
3,40.102790,10.230543,510.882922


- segment 0 ~ nhóm Loyal Customers

- segment 1 ~ nhóm At Risk

- segment 2 ~ nhóm Champions

- segment 3 ~ nhóm New Customers

In [12]:
# get true_lifetime_days for evaluation
rfm_c = rfm.merge(
    df_customers[["customer_id", "true_lifetime_days"]],
    on="customer_id",
    how="left"
)

In [13]:
rfm_c.groupby("segment")["true_lifetime_days"].mean()

segment
0    235.257143
1     88.853318
2    302.828829
3    162.342878
Name: true_lifetime_days, dtype: float64



-> KMeans cho thấy các cluster ứng với nhóm Champions/Loyal Customers có true lifetime cao hơn đáng kể, trong khi cluster ứng với nhóm At Risk có true lifetime ngắn nhất(khả năng churn cao nhất) 


In [14]:
rfm.to_parquet(
    DATA_PROCESSED / "rfm_features.parquet",
    index=False
)